# Match Outcome Model (1X2 and Either-Half)

This notebook trains the direct match-outcome models. Shared helpers from `football_prediction` handle loading, Elo, rolling form, calibration, metrics, and fixture state.

The models are one multiclass classifier for home win, draw, and away win, plus two binary classifiers for either-half outcomes. The final decision layer can skip matches when probabilities are not strong enough.


## 1. Configuration

Features are split into required core features and missing-tolerant features. Missing-tolerant features are mostly xG and head-to-head values, which do not exist for every row.


In [17]:
import warnings

import joblib
import numpy
import pandas
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss
from xgboost import XGBClassifier

from src.football_prediction.core import config
from src.football_prediction.data import loading as data
from src.football_prediction.features import rolling as features
from src.football_prediction.features.elo import add_elo_features
from src.football_prediction.features import state
from src.football_prediction.modeling import calibration, metrics

warnings.filterwarnings("ignore", category=FutureWarning)

project_root = config.PROJECT_ROOT
data_path = config.DATA_PATH
models_directory = config.MODELS_DIRECTORY
artifact_path = models_directory / "match_1x2_pred.joblib"

short_window = config.SHORT_WINDOW
long_window = config.LONG_WINDOW
long_window_minimum_matches = config.LONG_WINDOW_MINIMUM_MATCHES
xg_window = config.XG_WINDOW
xg_window_minimum_matches = config.XG_WINDOW_MINIMUM_MATCHES
minimum_team_matches = config.MINIMUM_TEAM_MATCHES
maximum_rest_days = config.MAXIMUM_REST_DAYS

initial_elo = config.INITIAL_ELO
elo_k = config.ELO_K
home_advantage = config.HOME_ADVANTAGE
validation_split_date = config.VALIDATION_SPLIT_DATE
test_split_date = config.TEST_SPLIT_DATE
random_seed = config.RANDOM_SEED
european_cup_names = config.EUROPEAN_CUP_NAMES
match_result_labels = config.MATCH_RESULT_LABELS

xgboost_parameters = {
    "n_estimators": 600,
    "learning_rate": 0.05,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "early_stopping_rounds": 50,
    "n_jobs": -1,
    "random_state": random_seed,
}
max_depth_candidates = [3, 4, 5]
min_child_weight_candidates = [1, 5, 10]

core_feature_names = [
    "elo_difference",
    "home_elo",
    "away_elo",
    "home_points_at_home",
    "away_points_at_away",
    "home_goals_scored_form",
    "away_goals_scored_form",
    "points_form_difference",
    "home_attack_vs_away_defence",
    "away_attack_vs_home_defence",
    "goal_difference_form_gap",
    "long_points_form_difference",
    "long_goal_difference_form_gap",
    "shots_on_target_difference",
    "shots_on_target_conceded_difference",
    "shot_accuracy_difference",
    "corners_difference",
    "possession_difference",
    "fouls_difference",
    "rest_days_difference",
    "is_european_cup",
    "h2h_matches_played",
]
nan_tolerant_feature_names = [
    "h2h_home_win_rate",
    "h2h_home_goal_difference",
    "home_xg_for_form",
    "away_xg_for_form",
    "home_xg_against_form",
    "away_xg_against_form",
    "home_finishing_luck_form",
    "away_finishing_luck_form",
]
feature_names = core_feature_names + nan_tolerant_feature_names

binary_target_names = ["home_wins_either_half", "away_wins_either_half"]
target_names = ["match_result"] + binary_target_names

pandas.set_option("display.max_columns", 80)
pandas.set_option("display.max_rows", 50)


## 2. Load Data

The loader reads the raw file, removes rows without required match stats, filters teams without enough history, and adds shared targets such as 1X2 and either-half labels.


In [18]:
matches, team_match_counts, team_filter_summary, load_summary = data.load_matches()

print("Cleaned matches:", matches.shape)
print("Data source:", data_path)
print("Date range:", matches["date_utc"].min().date(), "to", matches["date_utc"].max().date())

display(pandas.DataFrame([load_summary]))
display(
    matches["match_result"]
    .map(match_result_labels)
    .value_counts()
    .rename_axis("result")
    .rename("matches")
    .to_frame()
)


Cleaned matches: (14721, 36)
Data source: C:\PROJECTS\Python\ML\betting\datasets\rich_stats\league_matches_stats.csv
Date range: 2020-08-18 to 2026-05-30


,rows_loaded,rows_after_required_columns,rows_after_team_filter,rows_removed_for_required_columns,rows_removed_for_low_history_teams,remaining_teams,minimum_remaining_team_matches
0,15645,15627,14721,18,906,322,10


,matches
result,
home_win,6507
away_win,4640
draw,3574


## 3. Build Features

Each match is converted into team-perspective history rows. Rolling features use previous matches only, so a fixture never uses its own result or stats as input.


In [19]:
form_column_names = features.form_column_names()


def build_feature_table(match_frame):
    matches_with_elo, final_team_ratings = add_elo_features(match_frame)
    team_match_rows = features.make_team_match_rows(matches_with_elo)
    rolling_team_form = features.add_rolling_team_form(team_match_rows)
    feature_table = features.add_head_to_head_features(matches_with_elo, team_match_rows)

    for side_name in ["home", "away"]:
        side_form = rolling_team_form[rolling_team_form["venue"].eq(side_name)][
            ["match_id"] + form_column_names
        ].rename(columns={column_name: f"{side_name}_{column_name}" for column_name in form_column_names})
        feature_table = feature_table.merge(side_form, on="match_id", how="left")

    # Venue and scoring form.
    feature_table["home_points_at_home"] = feature_table["home_rolling_points_at_venue"]
    feature_table["away_points_at_away"] = feature_table["away_rolling_points_at_venue"]
    feature_table["home_goals_scored_form"] = feature_table["home_rolling_goals_scored"]
    feature_table["away_goals_scored_form"] = feature_table["away_rolling_goals_scored"]

    # Cross-matched form differences over the short window.
    feature_table["points_form_difference"] = (
        feature_table["home_rolling_points"] - feature_table["away_rolling_points"]
    )
    feature_table["home_attack_vs_away_defence"] = (
        feature_table["home_rolling_goals_scored"] - feature_table["away_rolling_goals_conceded"]
    )
    feature_table["away_attack_vs_home_defence"] = (
        feature_table["away_rolling_goals_scored"] - feature_table["home_rolling_goals_conceded"]
    )
    feature_table["goal_difference_form_gap"] = (
        feature_table["home_rolling_goal_difference"] - feature_table["away_rolling_goal_difference"]
    )
    feature_table["shots_on_target_difference"] = (
        feature_table["home_rolling_shots_on_target"] - feature_table["away_rolling_shots_on_target"]
    )
    feature_table["shots_on_target_conceded_difference"] = (
        feature_table["home_rolling_shots_on_target_conceded"]
        - feature_table["away_rolling_shots_on_target_conceded"]
    )
    feature_table["shot_accuracy_difference"] = (
        feature_table["home_rolling_shot_accuracy"] - feature_table["away_rolling_shot_accuracy"]
    )
    feature_table["corners_difference"] = (
        feature_table["home_rolling_corners"] - feature_table["away_rolling_corners"]
    )
    feature_table["possession_difference"] = (
        feature_table["home_rolling_possession"] - feature_table["away_rolling_possession"]
    )
    feature_table["fouls_difference"] = (
        feature_table["home_rolling_fouls"] - feature_table["away_rolling_fouls"]
    )
    feature_table["rest_days_difference"] = (
        feature_table["home_rest_days"] - feature_table["away_rest_days"]
    )

    # Slower-moving form over the long window.
    feature_table["long_points_form_difference"] = (
        feature_table["home_long_rolling_points"] - feature_table["away_long_rolling_points"]
    )
    feature_table["long_goal_difference_form_gap"] = (
        feature_table["home_long_rolling_goal_difference"]
        - feature_table["away_long_rolling_goal_difference"]
    )

    # Missing-tolerant xG form.
    feature_table["home_xg_for_form"] = feature_table["home_rolling_expected_goals_for"]
    feature_table["away_xg_for_form"] = feature_table["away_rolling_expected_goals_for"]
    feature_table["home_xg_against_form"] = feature_table["home_rolling_expected_goals_against"]
    feature_table["away_xg_against_form"] = feature_table["away_rolling_expected_goals_against"]
    feature_table["home_finishing_luck_form"] = feature_table["home_rolling_finishing_luck"]
    feature_table["away_finishing_luck_form"] = feature_table["away_rolling_finishing_luck"]

    return feature_table, team_match_rows, rolling_team_form, final_team_ratings


feature_table, team_match_rows, rolling_team_form, final_team_ratings = build_feature_table(matches)
model_data = feature_table.dropna(subset=core_feature_names).copy()

print("Feature build summary")
display(pandas.DataFrame([{
    "rows_before_dropping_no_history_rows": len(feature_table),
    "rows_after_dropping_no_history_rows": len(model_data),
    "rows_dropped": len(feature_table) - len(model_data),
    "core_features": len(core_feature_names),
    "missing_tolerant_features": len(nan_tolerant_feature_names),
    "total_features": len(feature_names),
}]))

print("\nMissing-tolerant feature availability (share of modelled rows with a value)")
display(
    model_data[nan_tolerant_feature_names]
    .notna()
    .mean()
    .rename("available")
    .round(3)
    .to_frame()
)

print("\nModeling table preview")
display(
    model_data[
        ["season", "competition", "date_utc", "home_team", "away_team"]
        + target_names
        + feature_names
    ].head()
)


Feature build summary


,rows_before_dropping_no_history_rows,rows_after_dropping_no_history_rows,rows_dropped,core_features,missing_tolerant_features,total_features
0,14721,12475,2246,22,8,30



Missing-tolerant feature availability (share of modelled rows with a value)


,available
h2h_home_win_rate,0.824
h2h_home_goal_difference,0.824
home_xg_for_form,0.581
away_xg_for_form,0.582
home_xg_against_form,0.581
away_xg_against_form,0.582
home_finishing_luck_form,0.581
away_finishing_luck_form,0.582



Modeling table preview


,season,competition,date_utc,home_team,away_team,match_result,home_wins_either_half,away_wins_either_half,elo_difference,home_elo,away_elo,home_points_at_home,away_points_at_away,home_goals_scored_form,away_goals_scored_form,points_form_difference,home_attack_vs_away_defence,away_attack_vs_home_defence,goal_difference_form_gap,long_points_form_difference,long_goal_difference_form_gap,shots_on_target_difference,shots_on_target_conceded_difference,shot_accuracy_difference,corners_difference,possession_difference,fouls_difference,rest_days_difference,is_european_cup,h2h_matches_played,h2h_home_win_rate,h2h_home_goal_difference,home_xg_for_form,away_xg_for_form,home_xg_against_form,away_xg_against_form,home_finishing_luck_form,away_finishing_luck_form
548,2020-2021,UEFA Europa League,2020-11-05 20:00:00,ac_milan,lille,2,0,1,20.738357,1579.115532,1558.377175,2.6,2.2,2.6,2.4,0.8,1.6,1.2,0.0,0.6,0.3,0.2,2.0,0.213888,-3.4,-3.8,1.4,0.0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
577,2020-2021,Ligue 1,2020-11-07 20:00:00,psg,rennes,0,1,0,53.058153,1569.541028,1516.482876,1.8,1.4,2.2,0.8,1.0,0.6,0.0,2.2,0.9,2.0,0.8,-0.6,-0.092199,1.6,1.6,-0.6,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
591,2020-2021,La Liga,2020-11-08 15:15:00,real_sociedad,granada_cf,0,1,0,19.446570,1561.667485,1542.220914,1.8,2.6,2.0,1.2,0.2,1.6,0.6,0.6,0.1,1.2,2.8,-0.6,0.092720,5.2,15.6,-2.6,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
603,2020-2021,Ligue 1,2020-11-20 20:00:00,monaco,psg,0,1,1,-62.561086,1516.085964,1578.647050,2.2,2.4,1.6,2.6,-1.0,1.2,1.2,-2.0,-0.7,-2.1,0.6,0.2,-0.060791,5.2,7.4,0.2,-1.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
614,2020-2021,La Liga,2020-11-21 15:15:00,villarreal,real_madrid,1,1,1,33.106964,1559.615307,1526.508342,3.0,2.0,2.4,2.6,0.6,0.4,2.2,1.4,0.4,0.6,-1.2,-1.8,0.023309,-2.0,2.8,-2.2,1.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Split by Time

The model is trained on older matches, tuned on the validation window, and evaluated once on the final test window.


In [20]:
training_matches = model_data[model_data["date_utc"] < validation_split_date].copy()
validation_matches = model_data[
    (model_data["date_utc"] >= validation_split_date)
    & (model_data["date_utc"] < test_split_date)
].copy()
test_matches = model_data[model_data["date_utc"] >= test_split_date].copy()


def result_rate(frame, result_code):
    return float(frame["match_result"].eq(result_code).mean()) if len(frame) else numpy.nan


split_summary = pandas.DataFrame([
    {
        "split": split_name,
        "rows": len(split_frame),
        "start": split_frame["date_utc"].min().date(),
        "end": split_frame["date_utc"].max().date(),
        "home_win_rate": result_rate(split_frame, 0),
        "draw_rate": result_rate(split_frame, 1),
        "away_win_rate": result_rate(split_frame, 2),
        "home_wins_either_half_rate": split_frame["home_wins_either_half"].mean(),
        "away_wins_either_half_rate": split_frame["away_wins_either_half"].mean(),
    }
    for split_name, split_frame in [
        ("training", training_matches),
        ("validation", validation_matches),
        ("test", test_matches),
    ]
])

print("Temporal three-way split summary")
print(f"Validation from {validation_split_date.date()}, test from {test_split_date.date()}")
display(split_summary.round(3))

training_feature_matrix = training_matches[feature_names]
validation_feature_matrix = validation_matches[feature_names]
test_feature_matrix = test_matches[feature_names]

Temporal three-way split summary
Validation from 2025-07-01, test from 2026-01-01


,split,rows,start,end,home_win_rate,draw_rate,away_win_rate,home_wins_either_half_rate,away_wins_either_half_rate
0,training,10072,2020-11-05,2025-05-31,0.439,0.246,0.315,0.579,0.469
1,validation,1251,2025-07-08,2025-12-30,0.474,0.225,0.301,0.595,0.432
2,test,1152,2026-01-01,2026-05-30,0.439,0.248,0.312,0.585,0.458


## 5. Baselines

The model must beat simple references before it is useful. The class-frequency baseline predicts training class rates, and the Elo-only baseline uses only the pre-match Elo gap.


In [21]:
training_class_rates = (
    training_matches["match_result"].value_counts(normalize=True).sort_index().to_numpy()
)


def class_frequency_probabilities(row_count):
    return numpy.tile(training_class_rates, (row_count, 1))


elo_only_model = LogisticRegression(max_iter=1000, random_state=random_seed)
elo_only_model.fit(training_matches[["elo_difference"]], training_matches["match_result"])

validation_baseline_table = pandas.DataFrame([
    {
        "model": "class_frequency",
        **metrics.evaluate_multiclass_probabilities(
            validation_matches["match_result"],
            class_frequency_probabilities(len(validation_matches)),
        ),
    },
    {
        "model": "elo_only_logistic",
        **metrics.evaluate_multiclass_probabilities(
            validation_matches["match_result"],
            elo_only_model.predict_proba(validation_matches[["elo_difference"]]),
        ),
    },
])

display(validation_baseline_table.round(4))


,model,accuracy,log_loss
0,class_frequency,0.4740,1.0534
1,elo_only_logistic,0.5396,0.9718


## 6. Train XGBoost Models

A small validation sweep chooses the tree depth and minimum child weight. The same selected shape is used for the two either-half classifiers.


In [22]:
sweep_records = []
best_sweep_record = None
best_sweep_model = None

for max_depth_value in max_depth_candidates:
    for min_child_weight_value in min_child_weight_candidates:
        candidate_model = XGBClassifier(
            **xgboost_parameters,
            objective="multi:softprob",
            eval_metric="mlogloss",
            max_depth=max_depth_value,
            min_child_weight=min_child_weight_value,
        )
        candidate_model.fit(
            training_feature_matrix,
            training_matches["match_result"],
            eval_set=[(validation_feature_matrix, validation_matches["match_result"])],
            verbose=False,
        )
        sweep_record = {
            "max_depth": max_depth_value,
            "min_child_weight": min_child_weight_value,
            "best_iteration": int(candidate_model.best_iteration),
            **metrics.evaluate_multiclass_probabilities(
                validation_matches["match_result"],
                candidate_model.predict_proba(validation_feature_matrix),
            ),
        }
        sweep_records.append(sweep_record)
        if best_sweep_record is None or sweep_record["log_loss"] < best_sweep_record["log_loss"]:
            best_sweep_record = sweep_record
            best_sweep_model = candidate_model

sweep_table = pandas.DataFrame(sweep_records).sort_values("log_loss").reset_index(drop=True)
print("Validation sweep (sorted by validation log-loss)")
display(sweep_table.round(4))

best_max_depth = best_sweep_record["max_depth"]
best_min_child_weight = best_sweep_record["min_child_weight"]
print(f"\nSelected configuration: max_depth={best_max_depth}, min_child_weight={best_min_child_weight}, "
      f"best_iteration={best_sweep_record['best_iteration']}")

trained_models = {"match_result": best_sweep_model}
for target_name in binary_target_names:
    binary_model = XGBClassifier(
        **xgboost_parameters,
        eval_metric="logloss",
        max_depth=best_max_depth,
        min_child_weight=best_min_child_weight,
    )
    binary_model.fit(
        training_feature_matrix,
        training_matches[target_name],
        eval_set=[(validation_feature_matrix, validation_matches[target_name])],
        verbose=False,
    )
    trained_models[target_name] = binary_model


def collect_model_metrics(models_by_target, split_frames):
    metric_records = []
    for split_name, split_frame in split_frames.items():
        split_features = split_frame[feature_names]
        metric_records.append({
            "target": "match_result",
            "split": split_name,
            "rows": len(split_frame),
            **metrics.evaluate_multiclass_probabilities(
                split_frame["match_result"],
                models_by_target["match_result"].predict_proba(split_features),
            ),
        })
        for target_name in binary_target_names:
            positive_probabilities = models_by_target[target_name].predict_proba(split_features)[:, 1]
            metric_records.append({
                "target": target_name,
                "split": split_name,
                "rows": len(split_frame),
                "accuracy": accuracy_score(
                    split_frame[target_name],
                    (positive_probabilities >= 0.5).astype(int),
                ),
                "log_loss": log_loss(split_frame[target_name], positive_probabilities, labels=[0, 1]),
            })
    return pandas.DataFrame(metric_records)


model_metric_table = collect_model_metrics(
    trained_models,
    {"training": training_matches, "validation": validation_matches},
)

print("\nModel accuracy and log-loss by split")
display(
    model_metric_table
    .pivot(index="target", columns="split", values="accuracy")
    .round(3)
    .rename(columns=lambda name: f"{name}_accuracy")
    .join(
        model_metric_table
        .pivot(index="target", columns="split", values="log_loss")
        .round(3)
        .rename(columns=lambda name: f"{name}_log_loss")
    )
)


Validation sweep (sorted by validation log-loss)


,max_depth,min_child_weight,best_iteration,accuracy,log_loss
0,3,5,66,0.5420,0.9720
1,4,5,69,0.5452,0.9720
2,3,10,66,0.5420,0.9723
3,4,1,71,0.5468,0.9725
4,4,10,64,0.5444,0.9727
5,3,1,66,0.5436,0.9729
6,5,5,62,0.5412,0.9746
7,5,10,56,0.5444,0.9749
8,5,1,62,0.5444,0.9754



Selected configuration: max_depth=3, min_child_weight=5, best_iteration=66

Model accuracy and log-loss by split


split,training_accuracy,validation_accuracy,training_log_loss,validation_log_loss
target,,,,
away_wins_either_half,0.656,0.645,0.614,0.630
home_wins_either_half,0.665,0.648,0.606,0.623
match_result,0.542,0.542,0.963,0.972


## 7. Calibrate Probabilities

Platt calibration learns a simple correction from validation predictions to validation outcomes. This improves the probability scale used by the decision layer.


In [23]:
raw_validation_probabilities = trained_models["match_result"].predict_proba(validation_feature_matrix)
match_result_calibrator = calibration.fit_multiclass_platt(
    raw_validation_probabilities,
    validation_matches["match_result"],
)


def apply_match_result_calibration(raw_probabilities, calibrator=None):
    if calibrator is None:
        calibrator = match_result_calibrator
    return calibration.apply_multiclass_platt(raw_probabilities, calibrator)


either_half_calibrators = {}
for target_name in binary_target_names:
    raw_positive_probabilities = trained_models[target_name].predict_proba(validation_feature_matrix)[:, 1]
    either_half_calibrators[target_name] = calibration.fit_binary_platt(
        raw_positive_probabilities,
        validation_matches[target_name],
    )

calibrated_validation_probabilities = apply_match_result_calibration(raw_validation_probabilities)
display(pandas.DataFrame([
    {
        "probabilities": "raw",
        **metrics.evaluate_multiclass_probabilities(
            validation_matches["match_result"],
            raw_validation_probabilities,
        ),
    },
    {
        "probabilities": "calibrated",
        **metrics.evaluate_multiclass_probabilities(
            validation_matches["match_result"],
            calibrated_validation_probabilities,
        ),
    },
]).round(4))


,probabilities,accuracy,log_loss
0,raw,0.5420,0.9720
1,calibrated,0.5436,0.9678


## 8. Decision Layer

The model can skip matches. Thresholds are chosen on validation to maximize precision while keeping at least 15 percent combined coverage where possible.


In [24]:
decision_labels = [
    "home",
    "away",
    "home win either half",
    "away win either half",
    "skip",
]


def predict_match_result_probabilities(match_result_model, feature_matrix, calibrated=True):
    probabilities = match_result_model.predict_proba(feature_matrix)
    if calibrated:
        probabilities = apply_match_result_calibration(probabilities)
    probability_frame = pandas.DataFrame(index=feature_matrix.index)
    for class_position, class_label in enumerate(match_result_model.classes_):
        result_name = match_result_labels[int(class_label)]
        probability_frame[f"{result_name}_probability"] = probabilities[:, class_position]
    return probability_frame


def score_model_probabilities(match_frame, models_by_target, calibrated=True):
    scored_matches = match_frame[[
        "season",
        "country",
        "competition",
        "date_utc",
        "home_team",
        "away_team",
        "match_result",
        "home_win",
        "draw",
        "away_win",
        "home_wins_either_half",
        "away_wins_either_half",
    ]].copy()
    feature_matrix = match_frame[feature_names]

    scored_matches = scored_matches.join(
        predict_match_result_probabilities(
            models_by_target["match_result"],
            feature_matrix,
            calibrated=calibrated,
        )
    )
    for target_name in binary_target_names:
        positive_probabilities = models_by_target[target_name].predict_proba(feature_matrix)[:, 1]
        if calibrated:
            positive_probabilities = calibration.apply_binary_calibration(
                positive_probabilities,
                either_half_calibrators[target_name],
            )
        scored_matches[f"{target_name}_probability"] = positive_probabilities
    return scored_matches.reset_index(drop=True)


def make_decision(
    home_win_probability,
    draw_probability,
    away_win_probability,
    home_wins_either_half_probability,
    away_wins_either_half_probability,
    thresholds,
):
    result_probabilities = {
        "home": home_win_probability,
        "draw": draw_probability,
        "away": away_win_probability,
    }
    ordered_results = sorted(result_probabilities.items(), key=lambda item: item[1], reverse=True)
    leading_result, leading_result_probability = ordered_results[0]
    result_probability_gap = leading_result_probability - ordered_results[1][1]

    if (
        leading_result in {"home", "away"}
        and leading_result_probability >= thresholds["minimum_outright_probability"]
        and result_probability_gap >= thresholds["minimum_outright_margin"]
    ):
        return {
            "decision": leading_result,
            "decision_probability": leading_result_probability,
        }

    either_half_probabilities = {
        "home win either half": home_wins_either_half_probability,
        "away win either half": away_wins_either_half_probability,
    }
    either_half_decision, either_half_probability = max(
        either_half_probabilities.items(),
        key=lambda item: item[1],
    )
    if either_half_probability >= thresholds["minimum_either_half_probability"]:
        return {
            "decision": either_half_decision,
            "decision_probability": either_half_probability,
        }

    return {
        "decision": "skip",
        "decision_probability": numpy.nan,
    }


def apply_decisions(scored_matches, thresholds):
    evaluation = scored_matches.copy()
    decision_records = pandas.DataFrame([
        make_decision(
            match_record.home_win_probability,
            match_record.draw_probability,
            match_record.away_win_probability,
            match_record.home_wins_either_half_probability,
            match_record.away_wins_either_half_probability,
            thresholds,
        )
        for match_record in evaluation.itertuples(index=False)
    ])
    evaluation = pandas.concat([evaluation.reset_index(drop=True), decision_records], axis=1)

    evaluation["bet_won"] = pandas.Series(pandas.NA, index=evaluation.index, dtype="boolean")
    evaluation.loc[evaluation["decision"].eq("home"), "bet_won"] = evaluation["home_win"].eq(1)
    evaluation.loc[evaluation["decision"].eq("away"), "bet_won"] = evaluation["away_win"].eq(1)
    evaluation.loc[evaluation["decision"].eq("home win either half"), "bet_won"] = (
        evaluation["home_wins_either_half"].eq(1)
    )
    evaluation.loc[evaluation["decision"].eq("away win either half"), "bet_won"] = (
        evaluation["away_wins_either_half"].eq(1)
    )
    return evaluation


def summarize_decisions(evaluation):
    summary_records = []
    total_matches = len(evaluation)

    for decision_label in decision_labels[:-1]:
        selected_decisions = evaluation[evaluation["decision"].eq(decision_label)]
        summary_records.append({
            "decision": decision_label,
            "picks": len(selected_decisions),
            "coverage": len(selected_decisions) / total_matches if total_matches else numpy.nan,
            "precision": (
                selected_decisions["bet_won"].astype(float).mean()
                if len(selected_decisions)
                else numpy.nan
            ),
        })

    selected_bets = evaluation[~evaluation["decision"].eq("skip")]
    summary_records.append({
        "decision": "combined slate",
        "picks": len(selected_bets),
        "coverage": len(selected_bets) / total_matches if total_matches else numpy.nan,
        "precision": selected_bets["bet_won"].astype(float).mean() if len(selected_bets) else numpy.nan,
    })
    skipped_matches = evaluation[evaluation["decision"].eq("skip")]
    summary_records.append({
        "decision": "skip",
        "picks": len(skipped_matches),
        "coverage": len(skipped_matches) / total_matches if total_matches else numpy.nan,
        "precision": numpy.nan,
    })
    return pandas.DataFrame(summary_records)


# Threshold tuning on the validation window.
validation_scored_matches = score_model_probabilities(validation_matches, trained_models)

outright_probability_candidates = [0.50, 0.55, 0.60, 0.65, 0.70]
outright_margin_candidates = [0.10, 0.15, 0.20, 0.25]
either_half_probability_candidates = [0.60, 0.65, 0.70, 0.75]
minimum_pick_coverage = 0.15

tuning_records = []
for outright_probability in outright_probability_candidates:
    for outright_margin in outright_margin_candidates:
        for either_half_probability in either_half_probability_candidates:
            candidate_thresholds = {
                "minimum_outright_probability": outright_probability,
                "minimum_outright_margin": outright_margin,
                "minimum_either_half_probability": either_half_probability,
            }
            candidate_evaluation = apply_decisions(validation_scored_matches, candidate_thresholds)
            selected_bets = candidate_evaluation[~candidate_evaluation["decision"].eq("skip")]
            tuning_records.append({
                **candidate_thresholds,
                "picks": len(selected_bets),
                "coverage": len(selected_bets) / len(candidate_evaluation),
                "precision": (
                    selected_bets["bet_won"].astype(float).mean()
                    if len(selected_bets)
                    else numpy.nan
                ),
            })

threshold_tuning_table = pandas.DataFrame(tuning_records)

eligible_thresholds = threshold_tuning_table[
    threshold_tuning_table["coverage"] >= minimum_pick_coverage
]
if eligible_thresholds.empty:
    eligible_thresholds = threshold_tuning_table
selected_threshold_row = (
    eligible_thresholds.sort_values(["precision", "coverage"], ascending=False).iloc[0]
)
decision_thresholds = {
    "minimum_outright_probability": float(selected_threshold_row["minimum_outright_probability"]),
    "minimum_outright_margin": float(selected_threshold_row["minimum_outright_margin"]),
    "minimum_either_half_probability": float(selected_threshold_row["minimum_either_half_probability"]),
}

print("Top threshold combinations by validation precision (coverage floor "
      f"{minimum_pick_coverage:.0%})")
display(
    eligible_thresholds.sort_values(["precision", "coverage"], ascending=False).head(10).round(3)
)

print("\nSelected decision thresholds:", decision_thresholds)
print("\nValidation decision summary with the selected thresholds")
display(summarize_decisions(apply_decisions(validation_scored_matches, decision_thresholds)).round(3))


Top threshold combinations by validation precision (coverage floor 15%)


,minimum_outright_probability,minimum_outright_margin,minimum_either_half_probability,picks,coverage,precision
67,0.7,0.10,0.75,275,0.220,0.771
71,0.7,0.15,0.75,275,0.220,0.771
75,0.7,0.20,0.75,275,0.220,0.771
79,0.7,0.25,0.75,275,0.220,0.771
66,0.7,0.10,0.70,412,0.329,0.755
70,0.7,0.15,0.70,412,0.329,0.755
74,0.7,0.20,0.70,412,0.329,0.755
78,0.7,0.25,0.70,412,0.329,0.755
65,0.7,0.10,0.65,602,0.481,0.731
69,0.7,0.15,0.65,602,0.481,0.731



Selected decision thresholds: {'minimum_outright_probability': 0.7, 'minimum_outright_margin': 0.1, 'minimum_either_half_probability': 0.75}

Validation decision summary with the selected thresholds


,decision,picks,coverage,precision
0,home,188,0.150,0.787
1,away,7,0.006,0.857
2,home win either half,47,0.038,0.766
3,away win either half,33,0.026,0.667
4,combined slate,275,0.220,0.771
5,skip,976,0.780,NaN


## 9. Test Evaluation

The test window is used only after model selection, calibration, and threshold tuning are finished.


In [25]:
raw_test_probabilities = trained_models["match_result"].predict_proba(test_feature_matrix)
calibrated_test_probabilities = apply_match_result_calibration(raw_test_probabilities)

test_comparison_table = pandas.DataFrame([
    {
        "model": "class_frequency",
        **metrics.evaluate_multiclass_probabilities(
            test_matches["match_result"],
            class_frequency_probabilities(len(test_matches)),
        ),
    },
    {
        "model": "elo_only_logistic",
        **metrics.evaluate_multiclass_probabilities(
            test_matches["match_result"],
            elo_only_model.predict_proba(test_matches[["elo_difference"]]),
        ),
    },
    {
        "model": "xgboost_raw",
        **metrics.evaluate_multiclass_probabilities(test_matches["match_result"], raw_test_probabilities),
    },
    {
        "model": "xgboost_calibrated",
        **metrics.evaluate_multiclass_probabilities(test_matches["match_result"], calibrated_test_probabilities),
    },
])
display(test_comparison_table.round(4))

result_label_order = [match_result_labels[result_code] for result_code in [0, 1, 2]]
test_result_predictions = calibrated_test_probabilities.argmax(axis=1)
display(
    pandas.DataFrame(
        classification_report(
            test_matches["match_result"],
            test_result_predictions,
            labels=[0, 1, 2],
            target_names=result_label_order,
            output_dict=True,
            zero_division=0,
        )
    ).transpose().round(3)
)

either_half_test_records = []
for target_name in binary_target_names:
    calibrated_positive = calibration.apply_binary_calibration(
        trained_models[target_name].predict_proba(test_feature_matrix)[:, 1],
        either_half_calibrators[target_name],
    )
    either_half_test_records.append({
        "target": target_name,
        "accuracy": accuracy_score(test_matches[target_name], (calibrated_positive >= 0.5).astype(int)),
        "log_loss": log_loss(test_matches[target_name], calibrated_positive, labels=[0, 1]),
        "positive_rate": float(test_matches[target_name].mean()),
    })
display(pandas.DataFrame(either_half_test_records).round(4))

test_scored_matches = score_model_probabilities(test_matches, trained_models)
test_evaluation_table = apply_decisions(test_scored_matches, decision_thresholds)
decision_summary_table = summarize_decisions(test_evaluation_table)
display(decision_summary_table.round(3))


,model,accuracy,log_loss
0,class_frequency,0.4392,1.0708
1,elo_only_logistic,0.5035,1.0170
2,xgboost_raw,0.5052,1.0088
3,xgboost_calibrated,0.5113,1.0165


,precision,recall,f1-score,support
home_win,0.534,0.794,0.639,506.000
draw,0.000,0.000,0.000,286.000
away_win,0.469,0.519,0.493,360.000
accuracy,0.511,0.511,0.511,0.511
macro avg,0.334,0.438,0.377,1152.000
weighted avg,0.381,0.511,0.434,1152.000


,target,accuracy,log_loss,positive_rate
0,home_wins_either_half,0.6389,0.6357,0.5851
1,away_wins_either_half,0.6189,0.6732,0.4583


,decision,picks,coverage,precision
0,home,204,0.177,0.686
1,away,13,0.011,0.769
2,home win either half,59,0.051,0.729
3,away win either half,28,0.024,0.679
4,combined slate,304,0.264,0.697
5,skip,848,0.736,NaN


## 10. Fixture Prediction

The fixture path rebuilds the same feature columns from a saved team-state store. Unknown team names are rejected because the model has no history for them.


In [34]:
trained_team_names = sorted(
    set(training_matches["home_team"]).union(training_matches["away_team"])
)


def build_team_state_store(results_frame):
    return state.build_team_state_store(results_frame)


def build_fixture_features(home_team, away_team, date, time="15:00",
                           european_cup=False, team_state_store=None):
    state.validate_known_teams(home_team, away_team, trained_team_names)
    if team_state_store is None:
        team_state_store = build_team_state_store(matches)

    fixture_date = pandas.to_datetime(f"{date} {time}", errors="coerce")
    if pandas.isna(fixture_date):
        fixture_date = pandas.to_datetime(date)

    elo_ratings = team_state_store["team_elo_ratings"]
    missing_elo_team_names = [
        team_name for team_name in [home_team, away_team] if team_name not in elo_ratings
    ]
    if missing_elo_team_names:
        raise ValueError(
            "Cannot build Elo features without stored ratings for: "
            + ", ".join(missing_elo_team_names)
        )
    home_elo = float(elo_ratings[home_team])
    away_elo = float(elo_ratings[away_team])

    def short_form(team_name, metric_name):
        return state.state_mean(team_state_store, team_name, "recent_matches", metric_name)

    def long_form(team_name, metric_name):
        return state.state_mean(
            team_state_store, team_name, "recent_matches", metric_name,
            window=long_window, minimum_matches=long_window_minimum_matches,
        )

    def xg_form(team_name, metric_name):
        return state.state_mean(
            team_state_store, team_name, "recent_matches", metric_name,
            window=xg_window, minimum_matches=xg_window_minimum_matches, required=False,
        )

    h2h_matches_played, h2h_home_win_rate, h2h_home_goal_difference = state.state_head_to_head(
        team_state_store, home_team, away_team,
    )

    fixture_feature_values = {
        "elo_difference": home_elo - away_elo,
        "home_elo": home_elo,
        "away_elo": away_elo,
        "home_points_at_home": state.state_mean(team_state_store, home_team, "recent_home_matches", "points"),
        "away_points_at_away": state.state_mean(team_state_store, away_team, "recent_away_matches", "points"),
        "home_goals_scored_form": short_form(home_team, "goals_scored"),
        "away_goals_scored_form": short_form(away_team, "goals_scored"),
        "points_form_difference": short_form(home_team, "points") - short_form(away_team, "points"),
        "home_attack_vs_away_defence": (
            short_form(home_team, "goals_scored") - short_form(away_team, "goals_conceded")
        ),
        "away_attack_vs_home_defence": (
            short_form(away_team, "goals_scored") - short_form(home_team, "goals_conceded")
        ),
        "goal_difference_form_gap": (
            short_form(home_team, "goal_difference") - short_form(away_team, "goal_difference")
        ),
        "long_points_form_difference": long_form(home_team, "points") - long_form(away_team, "points"),
        "long_goal_difference_form_gap": (
            long_form(home_team, "goal_difference") - long_form(away_team, "goal_difference")
        ),
        "shots_on_target_difference": (
            short_form(home_team, "shots_on_target") - short_form(away_team, "shots_on_target")
        ),
        "shots_on_target_conceded_difference": (
            short_form(home_team, "shots_on_target_conceded")
            - short_form(away_team, "shots_on_target_conceded")
        ),
        "shot_accuracy_difference": (
            short_form(home_team, "shot_accuracy") - short_form(away_team, "shot_accuracy")
        ),
        "corners_difference": short_form(home_team, "corners") - short_form(away_team, "corners"),
        "possession_difference": short_form(home_team, "possession") - short_form(away_team, "possession"),
        "fouls_difference": short_form(home_team, "fouls") - short_form(away_team, "fouls"),
        "rest_days_difference": (
            state.state_rest_days(team_state_store, home_team, fixture_date)
            - state.state_rest_days(team_state_store, away_team, fixture_date)
        ),
        "is_european_cup": int(european_cup),
        "h2h_matches_played": h2h_matches_played,
        "h2h_home_win_rate": h2h_home_win_rate,
        "h2h_home_goal_difference": h2h_home_goal_difference,
        "home_xg_for_form": xg_form(home_team, "expected_goals_for"),
        "away_xg_for_form": xg_form(away_team, "expected_goals_for"),
        "home_xg_against_form": xg_form(home_team, "expected_goals_against"),
        "away_xg_against_form": xg_form(away_team, "expected_goals_against"),
        "home_finishing_luck_form": xg_form(home_team, "finishing_luck"),
        "away_finishing_luck_form": xg_form(away_team, "finishing_luck"),
    }
    return pandas.DataFrame([fixture_feature_values])[feature_names]


def predict_fixture(home_team, away_team, date, time="15:00", european_cup=False,
                    team_state_store=None, models_by_target=None):
    if team_state_store is None:
        team_state_store = build_team_state_store(matches)
    if models_by_target is None:
        models_by_target = trained_models

    fixture_features = build_fixture_features(
        home_team,
        away_team,
        date,
        time=time,
        european_cup=european_cup,
        team_state_store=team_state_store,
    )

    result_probabilities = predict_match_result_probabilities(
        models_by_target["match_result"],
        fixture_features,
        calibrated=True,
    ).iloc[0].to_dict()
    probability_values = {**result_probabilities}
    for target_name in binary_target_names:
        raw_positive = float(models_by_target[target_name].predict_proba(fixture_features)[:, 1][0])
        probability_values[f"{target_name}_probability"] = float(
            calibration.apply_binary_calibration(
                numpy.array([raw_positive]), either_half_calibrators[target_name],
            )[0]
        )

    decision = make_decision(
        probability_values["home_win_probability"],
        probability_values["draw_probability"],
        probability_values["away_win_probability"],
        probability_values["home_wins_either_half_probability"],
        probability_values["away_wins_either_half_probability"],
        decision_thresholds,
    )

    return pandas.DataFrame([{
        "date": pandas.to_datetime(date).date(),
        "time": time,
        "home_team": home_team,
        "away_team": away_team,
        **probability_values,
        **decision,
    }])

predict_fixture("arsenal", "chelsea", "2026-10-01")


,date,time,home_team,away_team,home_win_probability,draw_probability,away_win_probability,home_wins_either_half_probability,away_wins_either_half_probability,decision,decision_probability
0,2026-10-01,15:00,arsenal,chelsea,0.797058,0.11675,0.086192,0.838937,0.32912,home,0.797058


## 11. Save Artifact

The artifact contains the trained models, calibrators, thresholds, features, team-state store, and evaluation tables needed to reproduce predictions later.


In [27]:
team_state_store = build_team_state_store(matches)

artifact = {
    "model_family": "XGBClassifier",
    "model_setup": "one multiclass 1X2 model plus two binary either-half models, "
                   "Platt-calibrated, with validation-tuned decision thresholds",
    "xgboost_parameters": xgboost_parameters,
    "selected_max_depth": best_max_depth,
    "selected_min_child_weight": best_min_child_weight,
    "trained_models": trained_models,
    "match_result_calibrator": match_result_calibrator,
    "either_half_calibrators": either_half_calibrators,
    "feature_names": feature_names,
    "core_feature_names": core_feature_names,
    "nan_tolerant_feature_names": nan_tolerant_feature_names,
    "target_names": target_names,
    "binary_target_names": binary_target_names,
    "match_result_labels": match_result_labels,
    "decision_thresholds": decision_thresholds,
    "short_window": short_window,
    "long_window": long_window,
    "long_window_minimum_matches": long_window_minimum_matches,
    "xg_window": xg_window,
    "xg_window_minimum_matches": xg_window_minimum_matches,
    "minimum_team_matches": minimum_team_matches,
    "maximum_rest_days": maximum_rest_days,
    "initial_elo": initial_elo,
    "elo_k": elo_k,
    "home_advantage": home_advantage,
    "european_cup_names": european_cup_names,
    "validation_split_date": validation_split_date,
    "test_split_date": test_split_date,
    "trained_team_names": trained_team_names,
    "team_state_store": team_state_store,
    "sweep_table": sweep_table,
    "model_metric_table": model_metric_table,
    "validation_baseline_table": validation_baseline_table,
    "test_comparison_table": test_comparison_table,
    "threshold_tuning_table": threshold_tuning_table,
    "decision_summary_table": decision_summary_table,
    "test_evaluation_table": test_evaluation_table,
    "team_filter_summary": team_filter_summary,
    "load_summary": load_summary,
    "data_path": str(data_path),
}

models_directory.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, artifact_path)
reloaded_artifact = joblib.load(artifact_path)

print(f"Saved artifact to {artifact_path}")
print("Reloaded artifact keys:", sorted(reloaded_artifact.keys()))
print("Reloaded model setup:", reloaded_artifact["model_setup"])
print("Reloaded feature count:", len(reloaded_artifact["feature_names"]))
print("Reloaded decision thresholds:", reloaded_artifact["decision_thresholds"])


Saved artifact to C:\PROJECTS\Python\ML\betting\models\match_1x2_pred.joblib
Reloaded artifact keys: ['binary_target_names', 'core_feature_names', 'data_path', 'decision_summary_table', 'decision_thresholds', 'either_half_calibrators', 'elo_k', 'european_cup_names', 'feature_names', 'home_advantage', 'initial_elo', 'load_summary', 'long_window', 'long_window_minimum_matches', 'match_result_calibrator', 'match_result_labels', 'maximum_rest_days', 'minimum_team_matches', 'model_family', 'model_metric_table', 'model_setup', 'nan_tolerant_feature_names', 'selected_max_depth', 'selected_min_child_weight', 'short_window', 'sweep_table', 'target_names', 'team_filter_summary', 'team_state_store', 'test_comparison_table', 'test_evaluation_table', 'test_split_date', 'threshold_tuning_table', 'trained_models', 'trained_team_names', 'validation_baseline_table', 'validation_split_date', 'xg_window', 'xg_window_minimum_matches', 'xgboost_parameters']
Reloaded model setup: one multiclass 1X2 model pl